# Mixture of Experts (MoE) - 실습 코드 2: MoE Load Balancing & 라우팅 분석

- Tutorial ID: `expand-moe`
- Tutorial: Mixture of Experts (MoE)
- Section ID: `expand-moe-code-2`
- Section: 실습 코드 2: MoE Load Balancing & 라우팅 분석


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: MoE Load Balancing & 라우팅 분석
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ── MoE Load Balancing 분석 ──
class MoEWithBalance(nn.Module):
    """Auxiliary Loss 기반 Load Balancing이 포함된 MoE"""
        def __init__(self, d_model, d_ff, num_experts, top_k=2, balance_weight=0.01):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.balance_weight = balance_weight
        
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.SiLU(),
                nn.Linear(d_ff, d_model),
            ) for _ in range(num_experts)
        ])
        self.shared_expert = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.SiLU(), nn.Linear(d_ff, d_model),
        )
    
        def forward(self, x):
        batch, seq, d = x.shape
        x_flat = x.view(-1, d)
        n_tokens = x_flat.size(0)
        
        # 라우팅
                logits = self.router(x_flat)
                probs = F.softmax(logits, dim=-1)
        top_k_probs, top_k_idx = torch.topk(probs, self.top_k, dim=-1)
        
        # Load Balancing Auxiliary Loss
        # f_i = 각 전문가에게 할당된 토큰 비율
        # P_i = 각 전문가의 평균 라우팅 확률
        # L_balance = N · Σ(f_i · P_i)  (균형 시 0에 가까움)
                expert_mask = torch.zeros(n_tokens, self.num_experts, device=x.device)
                for k in range(self.top_k):
            expert_mask.scatter_(1, top_k_idx[:, k:k+1], 1.0)
        
                f = expert_mask.mean(dim=0)       # (num_experts,) 할당 비율
        P = probs.mean(dim=0)             # (num_experts,) 평균 확률
                aux_loss = self.num_experts * torch.sum(f * P)
        
        # DeepSeek 스타일: bias 기반 균형 (auxiliary loss 대체)
        # router bias를 조정하여 균형 유지
        
        # 전문가 계산
                output = torch.zeros_like(x_flat)
                for i in range(self.num_experts):
                        mask = (top_k_idx == i).any(dim=-1)
            if mask.any():
                                expert_out = self.experts[i](x_flat[mask])
                # Top-K 가중치 적용
                                for k in range(self.top_k):
                                        k_mask = (top_k_idx[mask, k] == i)
                    if k_mask.any():
                        w = top_k_probs[mask, k][k_mask].unsqueeze(-1)
                                                idx = mask.nonzero(as_tuple=True)[0][k_mask]
                        output[idx] += w * expert_out[k_mask]
        
        # Shared Expert 추가
                output = output + self.shared_expert(x_flat)
        
        return output.view(batch, seq, d), aux_loss

# ── 라우팅 분석 도구 ──
def analyze_routing(moe_layer, x, top_k=2):
    """MoE 라우팅 패턴 분석"""
    batch, seq, d = x.shape
    x_flat = x.view(-1, d)
    
    with torch.no_grad():
                logits = moe_layer.router(x_flat)
                probs = F.softmax(logits, dim=-1)
        top_k_probs, top_k_idx = torch.topk(probs, top_k, dim=-1)
    
    # 전문가별 할당 토큰 수
    expert_counts = torch.zeros(moe_layer.num_experts)
        for k in range(top_k):
                for i in range(moe_layer.num_experts):
            expert_counts[i] += (top_k_idx[:, k] == i).sum().item()
    
    # 라우팅 다양성 (엔트로피)
    avg_entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1).mean().item()
    max_entropy = np.log(moe_layer.num_experts)
    
    print(f"=== MoE 라우팅 분석 ===")
    print(f"토큰 수: {x_flat.size(0)}, 전문가 수: {moe_layer.num_experts}, Top-K: {top_k}")
    print(f"\n전문가별 할당:")
        for i, count in enumerate(expert_counts):
        pct = count / (x_flat.size(0) * top_k) * 100
        bar = "█" * int(pct / 2)
        print(f"  E{i:>2}: {count:>5} 토큰 ({pct:>5.1f}%) {bar}")
    
    print(f"\n균형 점수: {1 - expert_counts.std() / expert_counts.mean():.3f} (1.0=완전 균형)")
    print(f"라우팅 엔트로피: {avg_entropy:.3f} / {max_entropy:.3f} "
          f"({avg_entropy/max_entropy*100:.1f}% 다양성)")
    
    return expert_counts

# ── 실험 ──
torch.manual_seed(42)
moe = MoEWithBalance(d_model=256, d_ff=1024, num_experts=8, top_k=2, balance_weight=0.01)

# 학습 전 라우팅 분석
x = torch.randn(4, 32, 256)
print("\n[학습 전]")
analyze_routing(moe, x)

# 약간의 학습 (라우팅 균형화)
optimizer = torch.optim.Adam(moe.parameters(), lr=1e-3)
for step in range(100):
        x = torch.randn(4, 32, 256)
        output, aux_loss = moe(x)
        main_loss = output.pow(2).mean()  # 더미 손실
        total_loss = main_loss + 0.01 * aux_loss
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

print("\n[학습 후 — Auxiliary Loss로 균형 개선]")
analyze_routing(moe, x)